In [61]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

# Basic Librariess
import numpy as np
import pandas as pd

import graceo_ot as got
from overtourism import *

import matplotlib.pyplot as plt
import igraph as ig
from igraph import Graph

### Other libraries
import os
import shutil
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

## Step 1: Graphs building

In [62]:
## Parameters for buidling dataset and extracting patterns
mode = "month" # "year" if summarize data by year
minSup = 0.1
max_size_patt = 5
city = "Lille"

## Reading files by city
if city == "Tampere":
    nodes_path = 'Data/Tampere/tripadvisorcorefua_Tampere_locations.csv'
    edges_path = 'Data/Tampere/tripadvisorcorefua_Tampere_circulationGraph.csv'
    likes_path = 'Data/Tampere/tripadvisorcorefua_Tampere_locations_evolution.csv'
    output_folder = "metrics_ot_" + city + "_" + mode
    stat_file = "metrics_ot_" + city + "_" + mode + "/stats_" + city + ".csv"
    freq_file = "metrics_ot_" + city + "_" + mode + "/freq_" + city + ".csv"
    patts_file = "metrics_ot_" + city + "_" + mode + "/patts_" + city + ".csv"
    patts_info_file = "metrics_ot_" + city + "_" + mode + "/patts_info_" + city + ".csv"
    metrics_file = "metrics_ot_" + city + "_" + mode + "/metrics_" + city + ".csv"
    regression_file = "metrics_ot_" + city + "_" + mode + "/regression_" + city + ".csv"
    map_file = "metrics_ot_" + city + "_" + mode + "/heatmap_" + city + ".pdf"
    #huff_file = "metrics_ot_" + city + "_" + mode + "/huff_" + city + ".pdf"
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
elif city == "Lille":
    nodes_path = 'Data/Lille/tripadvisorcorefua_Lille_locations.csv'
    edges_path = 'Data/Lille/tripadvisorcorefua_Lille_circulationGraph.csv'
    likes_path = 'Data/Lille/tripadvisorcorefua_Lille_locations_evolution.csv'
    output_folder = "metrics_ot_" + city + "_" + mode
    stat_file = "metrics_ot_" + city + "_" + mode + "/stats_" + city + ".csv"
    freq_file = "metrics_ot_" + city + "_" + mode + "/freq_" + city + ".csv"
    patts_file = "metrics_ot_" + city + "_" + mode + "/patts_" + city + ".csv"
    patts_info_file = "metrics_ot_" + city + "_" + mode + "/patts_info_" + city + ".csv"
    metrics_file = "metrics_ot_" + city + "_" + mode + "/metrics_" + city + ".csv"
    regression_file = "metrics_ot_" + city + "_" + mode + "/regression_" + city + ".csv"
    map_file = "metrics_ot_" + city + "_" + mode + "/heatmap_" + city + ".pdf"
    #huff_file = "metrics_ot_" + city + "_" + mode + "/huff_" + city + ".pdf"
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
elif city == "Barcelona":
    nodes_path = 'Data/Barcelona/tripadvisorcorefua_Barcelona_locations.csv'
    edges_path = 'Data/Barcelona/tripadvisorcorefua_Barcelona_circulationGraph.csv'
    likes_path = 'Data/Barcelona/tripadvisorcorefua_Barcelona_locations_evolution.csv'
    output_folder = "metrics_ot_" + city + "_" + mode
    stat_file = "metrics_ot_" + city + "_" + mode + "/stats_" + city + ".csv"
    freq_file = "metrics_ot_" + city + "_" + mode + "/freq_" + city + ".csv"
    patts_file = "metrics_ot_" + city + "_" + mode + "/patts_" + city + ".csv"
    patts_info_file = "metrics_ot_" + city + "_" + mode + "/patts_info_" + city + ".csv"
    metrics_file = "metrics_ot_" + city + "_" + mode + "/metrics_" + city + ".csv"
    regression_file = "metrics_ot_" + city + "_" + mode + "/regression_" + city + ".csv"
    map_file = "metrics_ot_" + city + "_" + mode + "/heatmap_" + city + ".pdf"
    #huff_file = "metrics_ot_" + city + "_" + mode + "/huff_" + city + ".pdf"
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

## Reading files before building graphs
fields_nodes = ['id', 'nom', 'latitude', 'longitude', 'typeR']
fields_edges = ['gid_from', 'gid_to', 'year', 'month', 'country', 'NbPerMaxDurationDays_14']
vertices, edges = got.read_files(nodes_path, edges_path, fields_nodes, fields_edges)
likes = pd.read_csv(likes_path,sep=";")
likes.rename(columns={"idplace": "id"}, inplace = True)

In [63]:
print("Folder path for storing the output results --> ", output_folder)

Folder path for storing the output results -->  metrics_ot_Lille_month


In [64]:
vertices.head()

,id,nom,latitude,longitude,typeR
0,196912,Mercure Lille Centre Grand Place,50.63817,3.065553,H
1,196913,Mercure Lille Roubaix Grand Hotel,50.69226,3.172287,H
2,196916,Ibis Lille Centre Gares,50.63469,3.070875,H
3,196917,Ibis Styles Lille Centre Gare Beffroi,50.63252,3.067851,H
4,196918,Ibis Styles Lille Centre Grand Place,50.63760,3.066830,H


In [65]:
edges.head()

,source,target,year,month,country,NbPerMaxDurationDays_14
1,1323722,3365098,2017,10,France,1
5,12379014,12379014,2017,4,Netherlands,1
6,12379014,13143832,2020,3,France,1
8,12379014,13143832,2019,11,France,1
9,12379014,17343293,2020,10,France,1


In [66]:
likes.head()

,id,year,month,rating,NB
0,196912,2013,1,4.000000,12
1,196912,2014,1,3.750000,16
2,196912,2015,1,4.363636,22
3,196912,2016,1,3.761905,21
4,196912,2017,1,3.944444,18


In [67]:
print("Min year --> ", min(edges["year"]))
print("Max year --> ", max(edges["year"]))

Min year -->  2013
Max year -->  2020


In [68]:
### Building graphs by year/month
## Here, sub-graph was buid for each month/year
## Then, a couple of vertices exists if there is a connecting edge

lst_graphs = got.build_graphs_by_time(vertices, edges, likes, mode=mode)

In [69]:
print("Number of graphs (by month/year) ---> ", len(lst_graphs))

Number of graphs (by month/year) --->  96


In [70]:
## Getting statistics by all graphs

df_stats = got.summarize_graphs(lst_graphs, stat_file)
print(df_stats)

   max_vertices  min_vertices  mean_vertices  max_edges  min_edges  mean_edges
0           646            38         419.29        781         20       456.3


## Step 2: Sub-graphs patterns mining with PAMI

In [71]:
## This dic mat the original IDs and the ID for canonical representation performed by gSpan

graphs_gspan_in_filename = "graphs_gspam.txt"
mapping = got.write_gspan_input(lst_graphs, graphs_gspan_in_filename)

In [72]:
## Extracting frequent subgraphs for thereshold and a max pettens size

graphs_gspan_out_filename = 'frequentSupgraphs.txt'

got.subgraph_pattens_mining(minSup, graphs_gspan_in_filename, graphs_gspan_out_filename, max_size_patt)

't # 0 * 85\nv 0 269814\nv 1 269817\ne 0 1 1\nt # 1 * 66\nv 0 269814\nv 1 269817\nv 2 269820\ne 0 1 1\ne 0 2 1\nt # 2 * 15\nv 0 269814\nv 1 269817\nv 2 269820\nv 3 628635\ne 0 1 1\ne 0 2 1\ne 0 3 1\nt # 3 * 10\nv 0 269814\nv 1 269817\nv 2 269820\nv 3 628635\nv 4 793722\ne 0 1 1\ne 0 2 1\ne 0 3 1\ne 0 4 1\nt # 4 * 36\nv 0 269814\nv 1 269817\nv 2 269820\nv 3 793722\ne 0 1 1\ne 0 2 1\ne 0 3 1\nt # 5 * 15\nv 0 269814\nv 1 269817\nv 2 269820\nv 3 793722\nv 4 801094\ne 0 1 1\ne 0 2 1\ne 0 3 1\ne 0 4 1\nt # 6 * 12\nv 0 269814\nv 1 269817\nv 2 269820\nv 3 793722\nv 4 734546\ne 0 1 1\ne 0 2 1\ne 0 3 1\ne 3 4 1\nt # 7 * 18\nv 0 269814\nv 1 269817\nv 2 269820\nv 3 793722\nv 4 801094\ne 0 1 1\ne 0 2 1\ne 0 3 1\ne 3 4 1\nt # 8 * 10\nv 0 269814\nv 1 269817\nv 2 269820\nv 3 793722\nv 4 810939\ne 0 1 1\ne 0 2 1\ne 0 3 1\ne 3 4 1\nt # 9 * 10\nv 0 269814\nv 1 269817\nv 2 269820\nv 3 793722\nv 4 3536508\ne 0 1 1\ne 0 2 1\ne 0 3 1\ne 3 4 1\nt # 10 * 18\nv 0 269814\nv 1 269817\nv 2 269820\nv 3 793722\nv 4 

In [81]:
## Recovering patterns stored by gSpan
## Saving the frequent pattens into a output_folder/file.txt

pattern_file = "frequentSupgraphs.txt"
src_path = pattern_file
dst_path = output_folder+"/"+pattern_file
shutil.copy(src_path, dst_path)

patterns = got.parse_gspan_patterns(pattern_file)

In [82]:
print("Total of frequent sub-graphs --> ", len(patterns))

Total of frequent sub-graphs -->  1633


In [83]:
## Creating lables for visualization purposes (not necesary)

label_x_axis = got.creation_label_x_axis(edges, mode)
print("Example of x-labels --> ", label_x_axis[0:10])

Example of x-labels -->  ['01-2013', '02-2013', '03-2013', '04-2013', '05-2013', '06-2013', '07-2013', '08-2013', '09-2013', '10-2013']


In [84]:
## We can also keep with the maximal frequent sub-graphs 

#max_patterns = got.find_max_subgraphs(patterns)
#print(f"{len(max_patterns)} maximal subgraphs found")

## Step 3: Adding features to the original monthly graphs

In [85]:
## Showing the number of graphs 

print("Number of graphs --> ", len(lst_graphs))

Number of graphs -->  96


In [86]:
## Showing the number of nodes for a specific graph in lst_graphs

print("Number of edges for specific graph --> ", lst_graphs[5].vcount())

Number of edges for specific graph -->  96


In [87]:
## Calculating the degree and the betweeness centrality

for g in lst_graphs:
    got.add_degree_and_betweenness(g)

In [88]:
## Calculating the MCA Huff model

criteria_attrs = ['indegree']
for g in lst_graphs:
    #g = got.compute_mca_huff(g, criteria_attrs, alpha=1, beta=1) ## MCA Huff metric
    g = got.attract_huff(g) ## Classical Huff metric

In [89]:
## Adding NB and Rating to graph nodes

got.add_rating_graphs(lst_graphs, likes)

In [90]:
#for v in lst_graphs[0].vs:
#    print(v)

In [91]:
## Buiding dataframes of vertices from original graphs 

lst_nodes_df = []
for i in range(len(lst_graphs)):
    name = 'df_nodes_'+str(i)
    globals()[name] = pd.DataFrame([{attr: v[attr] for attr in v.attributes()} for v in lst_graphs[i].vs])
    lst_nodes_df.append(pd.DataFrame([{attr: v[attr] for attr in v.attributes()} for v in lst_graphs[i].vs]))

In [92]:
print("Number of datafranes (months) --> ",len(lst_nodes_df))

Number of datafranes (months) -->  96


In [93]:
## Showing the number of nodes for a speccific month in list of dataframes (just to validate)

len(lst_nodes_df[5])

96

In [94]:
lst_nodes_df[5].head()

,id,nom,latitude,longitude,typeR,rating,NB,indegree,bcentrality,mca_huff
0,196912,Mercure Lille Centre Grand Place,50.638170,3.065553,H,3.812500,16.0,1,0.0,0.007080
1,196920,Ibis Lille Centre Grand Place,50.638317,3.062930,H,3.818182,11.0,2,42.0,0.016669
2,209264,Carrefour,50.641920,3.085010,A,3.000000,1.0,1,0.0,0.003378
3,238906,Hotel de la Paix,50.636170,3.065797,H,5.000000,1.0,1,0.0,0.007478
4,251421,Ibis Styles Lille Aeroport,50.580463,3.091446,H,3.000000,10.0,1,0.0,0.003740


In [95]:
## Test for having the data type of the feature "id"

print(lst_nodes_df[3]["id"][0])
print(type(lst_nodes_df[3]["id"][0]))

196912
<class 'numpy.int64'>


## Step 4: Matching the frequent subgraph extracted by gSpan with original data

In [96]:
## Recovering data from file created by gSpan
## This dataframe contains a list of nodes and the support for each pattern

df_patterns = got.parse_gspan_output(pattern_file)

In [97]:
## Storing gSpan patterns into a file

df_patterns.to_csv(patts_file, index=False)

In [98]:
df_patterns.sort_values(by="support", ascending=False).head(10)

,nb,vertices,support
0,0,"[269814, 269817]",85
1,563,"[269817, 269820]",72
2,113,"[269814, 269817, 269820]",71
3,334,"[269814, 269820]",70
4,1,"[269814, 269817, 269820]",66
5,357,"[269814, 269820, 269817]",58
6,114,"[269814, 269817, 269820]",57
7,511,"[269814, 793722]",47
8,78,"[269814, 269817, 793722]",46
9,637,"[269820, 793722]",45


In [99]:
## Showing a pattern by pattern_id -> nb

pattern_id = 0

df_patterns[df_patterns['nb'] == pattern_id]

,nb,vertices,support
0,0,"[269814, 269817]",85


In [100]:
## Adding other features to nodes and buiding a dataframe

df_vertices_patt = got.extend_vertices_patt(df_patterns, vertices)

In [101]:
## Storing the dataframe into a file

df_vertices_patt.to_csv(patts_info_file, index=False)

In [102]:
df_vertices_patt.head(10)

,id,nom,latitude,longitude,typeR,nb,support
0,269814,Grande Place,50.636970,3.063670,A,0,85
1,269817,Vieille Bourse,50.637730,3.064200,A,0,85
2,269817,Vieille Bourse,50.637730,3.064200,A,563,72
3,269820,Palais des Beaux-Arts de Lille,50.630672,3.062632,A,563,72
4,269814,Grande Place,50.636970,3.063670,A,113,71
5,269817,Vieille Bourse,50.637730,3.064200,A,113,71
6,269820,Palais des Beaux-Arts de Lille,50.630672,3.062632,A,113,71
7,269814,Grande Place,50.636970,3.063670,A,334,70
8,269820,Palais des Beaux-Arts de Lille,50.630672,3.062632,A,334,70
9,269814,Grande Place,50.636970,3.063670,A,1,66


In [103]:
## Query for searching patterns containing types of places (A, R, H)

got.filter_patts_by_types(df_vertices_patt, patt_type={'R', 'A', 'A'}).head(5)

,id,nom,latitude,longitude,typeR,nb,support
18,269814,Grande Place,50.63697,3.063670,A,511,47
19,793722,La Chicoree,50.63596,3.063038,R,511,47
20,269814,Grande Place,50.63697,3.063670,A,78,46
21,269817,Vieille Bourse,50.63773,3.064200,A,78,46
22,793722,La Chicoree,50.63596,3.063038,R,78,46


In [104]:
## Selecting a specific pattern to show the vertices

df_vertices_patt[df_vertices_patt['nb'] == 0]

,id,nom,latitude,longitude,typeR,nb,support
0,269814,Grande Place,50.63697,3.06367,A,0,85
1,269817,Vieille Bourse,50.63773,3.06420,A,0,85


In [105]:
## Looking for a specific ID (in patterns) into the vertices datafrae

mode_id_test = df_vertices_patt["id"].head(1)
print("First id node of patterns (for testing) --> ", int(mode_id_test))

vertices[vertices["id"] == int(mode_id_test)]

First id node of patterns (for testing) -->  269814


,id,nom,latitude,longitude,typeR
51,269814,Grande Place,50.63697,3.06367,A


In [106]:
#lst_graphs[0].vs.find(id = 592831)

In [107]:
#metrics = ['indegree', 'bcentrality', 'mca_huff', 'rating', 'NB']
#metrics = ['mca_huff']
#got.plot_metrics(lst_patterns[pattern_number], metrics, label_x_axis, output_folder, mode, style = ['science', 'retro'])

## Step 5: Ploting heatmap

In [108]:
## Creatin dataframe for visualizing heatmaps

freq_nodes_patt = got.frequence_by_vertice_patt(df_vertices_patt)

In [109]:
freq_nodes_patt.head(10)

,id,nom,latitude,longitude,frequency
0,269814,Grande Place,50.636970,3.063670,1236
1,269820,Palais des Beaux-Arts de Lille,50.630672,3.062632,1082
2,269817,Vieille Bourse,50.637730,3.064200,976
3,793722,La Chicoree,50.635960,3.063038,774
4,247498,"LaM Lille Métropole Musée d'Art Moderne, d'Art...",50.637512,3.150333,290
5,1381296,Meert,50.637680,3.061862,233
6,209260,Westfield Euralille,50.637753,3.072719,194
7,734546,Place Rihour,50.635643,3.062659,194
8,801094,Estaminet Chez la vieille,50.642830,3.066893,169
9,938725,Brasserie de la Paix,50.635815,3.063069,113


In [110]:
got.density_map_plotly(freq_nodes_patt, map_file, 12, city)

## Step 6: Visualization of metrics

### Step 6.1 - Visualization of rating and NB

In [111]:
likes.head()

,id,year,month,rating,NB
0,196912,2013,1,4.000000,12
1,196912,2014,1,3.750000,16
2,196912,2015,1,4.363636,22
3,196912,2016,1,3.761905,21
4,196912,2017,1,3.944444,18


In [112]:
## Normalization of NB and ratings

likes_norm = got.normalize_ratings(likes)

In [113]:
likes_norm.to_csv(metrics_file, index=False)
likes_norm.head()

,id,year,month,rating,NB,NB_acc,add_rating,norm_rating,aux,norm_nb
0,196912,2013,1,4.0,12,12.0,48.0,4.000000,616,1.948052
1,196912,2013,2,4.0,5,17.0,68.0,4.000000,580,0.862069
2,196912,2013,3,3.8,5,22.0,87.0,3.954545,651,0.768049
3,196912,2013,4,3.3,10,32.0,120.0,3.750000,696,1.436782
4,196912,2013,5,3.8,10,42.0,158.0,3.761905,757,1.321004


In [114]:
## Looking the meatrics for a specific node (the first one) in the frequent subgraphs dataframe 

mode_id_test = df_vertices_patt["id"].head(1)
print("First id node of patterns (for testing) --> ", int(mode_id_test))
likes_norm[likes_norm["id"]== int(mode_id_test)].head()

First id node of patterns (for testing) -->  269814


,id,year,month,rating,NB,NB_acc,add_rating,norm_rating,aux,norm_nb
2085,269814,2013,1,4.266667,15,15.0,64.0,4.266667,616,2.435065
2086,269814,2013,2,4.333333,9,24.0,103.0,4.291667,580,1.551724
2087,269814,2013,3,4.307692,13,37.0,159.0,4.297297,651,1.996928
2088,269814,2013,4,4.111111,9,46.0,196.0,4.260870,696,1.293103
2089,269814,2013,5,4.666667,9,55.0,238.0,4.327273,757,1.188904


In [115]:
## Configuring the visualization function

nb_places_to_viz = 2 # Number of places to visualize (2x2 matrix)

lst_nodes_plot = freq_nodes_patt["id"].head(nb_places_to_viz).to_list()

#lst_nodes_plot = [269814, 269820, 269817, 6208344] ## Visualize personalized nodes
#lst_nodes_plot = [661108, 941703] ## Visualize personalized nodes
lst_features_plot = ['norm_rating','norm_nb'] 

In [116]:
lst_nodes_plot

[269814, 269820]

In [117]:
print("List of nodes to visualize --> ", lst_nodes_plot)

List of nodes to visualize -->  [269814, 269820]


In [118]:
## Ploting the normalized nb and rating for the first four frequent places in patterns
## Ploting lineplots by node_id and a figure containing all plots (2x2 matrix)

for i in range(len(lst_nodes_plot)):
    got.plot_likes_evolution(likes_norm, lst_nodes_plot[i], lst_features_plot, output_folder, style = ['science', 'ieee'])

got.plot_likes_evol_multi(likes_norm, lst_nodes_plot, lst_features_plot, output_folder, style = ['science', 'ieee'])

### Step 6.2 - Visualization of Huff attractiveness

In [119]:
## Uncomment for testing the visualization 

#mode_id_test = df_vertices_patt["id"].head(1)
#print("First id node of patterns (for testing) --> ", int(mode_id_test))
#got.plot_mca_huff_trend(df_huff_serie, mode_id_test, ["norm_huff","log_regre"], label_x_axis, output_folder, style = ['science', 'ieee'])

In [120]:
## Ploting the MCA Huff attractiveness for the first four frequent places in patterns
## Ploting lineplots by node_id and a figure containing all plots (2x2 matrix)

attr_to_viz = ["norm_huff","log_regre"]

for i in range(len(lst_nodes_plot)):
    df_huff_serie = got.extract_mca_huff_series(lst_graphs, lst_nodes_plot[i], output_folder)
    got.plot_mca_huff_trend(df_huff_serie, lst_nodes_plot[i], attr_to_viz, label_x_axis, output_folder, style = ['science', 'ieee'])

got.plot_mca_huff_trend_multi(lst_graphs, lst_nodes_plot, attr_to_viz, label_x_axis, output_folder, style = ['science', 'ieee'])

In [121]:
df_vertices_patt["id"].unique()

array([  269814,   269817,   269820,   793722,   247498,   209260,
        1381296,   801094,   938725,  1740729,  1775539,   734546,
        5584607, 13387436,   196912,  4242962,  3347481,  6208344,
        6847194,  2558192,  2273093,  1026189,   196917,  2530949,
        2035312,   196916,  4408913,   810939,   628635,  2039935,
         198290,  8814343,  8817383,  1322553,  2291108,  1324334,
         196918,  2272480,  3536508,   781110,  2509707,   795570,
         300759,   275111,  8564361,  8758569,  1198856,   812062,
        1124428,   814600, 11670679, 12442472,   626951,  1393100,
        2272417,   795212,   909317,  3737668,  3944612,   803286,
       10317510, 10317530,   196920,  1089886, 10163609,  1236485,
         968557,   783726,  2719662,  3526029,  1906444,   277055,
        2614547,  1788496,  6211236,   877516,  6209227,  6166986,
        1008232,  1332161,  1237411,  1380803,  1239099,  9707431,
       11666532,  6924198,  7097423,  4991766,  2061883,  2401

In [122]:
all_lst_nodes = freq_nodes_patt["id"].to_list()
print(all_lst_nodes)


[269814, 269820, 269817, 793722, 247498, 1381296, 209260, 734546, 801094, 938725, 196912, 2273093, 196917, 1026189, 196916, 2558192, 2035312, 198290, 795570, 196918, 275111, 1322553, 2039935, 628635, 2530949, 812062, 810939, 5584607, 781110, 3536508, 13387436, 4242962, 1740729, 1198856, 909317, 277055, 1124428, 1324334, 196920, 1906444, 795212, 1775539, 4408913, 3347481, 2509707, 6208344, 2291108, 2272417, 803286, 3526029, 968557, 4991766, 877516, 300759, 2272480, 6847194, 2234190, 6166986, 1237411, 813222, 6924198, 968102, 2061883, 8814343, 10317510, 8817383, 3737668, 1606349, 781881, 1072502, 4462758, 4736631, 6672509, 5607118, 3747537, 5416584, 6016790, 794954, 3431136, 4408912, 7762248, 7928400, 1393095, 3746256, 3539535, 801700, 3722878, 10232584, 3784521, 3944612, 2401294, 7097423, 8564361, 8758569, 814600, 11670679, 12442472, 626951, 1393100, 10317530, 1089886, 10163609, 1236485, 783726, 2719662, 2614547, 1788496, 6211236, 6209227, 1008232, 1332161, 1380803, 1239099, 9707431, 11

In [123]:
lst_aux = []
for i in range(len(all_lst_nodes)):
    aux = got.compute_huff_regression(lst_graphs, all_lst_nodes[i], output_folder)
    lst_aux.append(aux)
df_huff_log = pd.DataFrame(lst_aux)
df_huff_log = df_huff_log.set_axis(['Node_id', 'nom', 'coef', 'inter'], axis=1)
df_huff_log.to_csv(regression_file)

In [124]:
df_huff_log.sort_values(by=['coef'], ascending=False)

,Node_id,nom,coef,inter
45,6208344,Bloempot,0.002413,-0.002476
65,8817383,Team Break Lille,0.002221,-0.004657
55,6847194,Rouge Barre,0.001806,-0.002342
63,8814343,John Doe Escape Game,0.001694,-0.003279
95,11670679,"Hotel L'Arbre Voyageur, BW Premier Collection",0.001561,-0.003728
...,...,...,...,...
36,1124428,L'Ecume des Mers,-0.004869,0.023572
0,269814,Grande Place,-0.006281,0.043557
16,2035312,Office de Tourisme de Lille,-0.006501,0.031582
1,269820,Palais des Beaux-Arts de Lille,-0.007341,0.040837
